In [2]:
import warnings
warnings.filterwarnings("ignore")

import torch 
import torch.nn.functional as F

In [1]:

import torch
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel
import gradio as gr
from PIL import Image

c:\Program Files\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nikha\AppData\Roaming\Python\Python311\site-packages\diffusers\models\transformers\transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
C:\Users\nikha\AppData\Roaming\Python\Python311\site-packages\diffusers\models\transformers\transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


In [ ]:
model_id = "openai/clip-vit-base-patch32"

tokenizer = CLIPTokenizer.from_pretrained(model_id)
text_encoder = CLIPTextModel.from_pretrained(model_id)

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_text_embedding(prompt):
  tokens = preprocess_text(prompt)

  with torch.no_grad():
    embedding = text_encoder(**tokens)
    return embedding.last_hidden_state

In [ ]:
def generate_image(prompt):

    # create CLIP embeddings
    embeddings = get_text_embedding(prompt)

    # generate image using Stable Diffusion
    image = pipe(prompt).images[0]

    return image

In [ ]:
def interface(prompt):

    image = generate_image(prompt)

    return image


demo = gr.Interface(
    fn=interface,
    inputs=gr.Textbox(label="Enter Text Prompt"),
    outputs=gr.Image(label="Generated Image"),
    title="Text to Image Generator",
    description="Stable Diffusion + CLIP + Hugging Face"
)

demo.launch()